In [2]:
import os
BASE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
PARENT_DIR = os.path.dirname(os.path.dirname(BASE_DIR))
HTML_FILE = os.path.join(PARENT_DIR, "data","HTML", "test.html")

In [63]:
from pathlib import Path
from bs4 import BeautifulSoup

def parse_table(table) -> str:
    rows = []

    for row in table.find_all("tr"):

        cells = row.find_all(["th", "td"])

        values = [cell.get_text(" ", strip=True) for cell in cells]

        rows.append(" | ".join(values))

    return "\n".join(rows)

def parse_html(file_path: str) -> list[dict]:
    html = Path(file_path).read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "lxml")
    document_title = soup.title.get_text(strip=True)
    elements = []

    for element in soup.body.find_all(['h1','h2','h3','p','table','pre','code']):
        if element.name == "table":
            text = parse_table(element)
        else:
            text = element.get_text(" ", strip=True)
        elements.append({
            "tag":element.name,
            "text":text
            })

    return {
            "title": document_title,
            "elements": elements
        }


In [64]:
result = parse_html(file_path=HTML_FILE)
print(result)
for content in result["elements"]:
    print(content)

{'title': 'Payment API Documentation', 'elements': [{'tag': 'h1', 'text': 'Payment Service'}, {'tag': 'p', 'text': 'The Payment Service provides APIs for processing customer payments.'}, {'tag': 'h2', 'text': 'Authentication'}, {'tag': 'p', 'text': 'The Payment API uses OAuth2 authentication.\nAccess tokens expire after 60 minutes.'}, {'tag': 'h2', 'text': 'Error Handling'}, {'tag': 'p', 'text': 'HTTP 401 indicates an invalid or expired token.'}, {'tag': 'p', 'text': 'HTTP 429 indicates that the rate limit has been exceeded.'}, {'tag': 'h2', 'text': 'Retry Policy'}, {'tag': 'p', 'text': 'Requests returning HTTP 429 can be retried using exponential backoff.'}, {'tag': 'h2', 'text': 'Payment Status Codes'}, {'tag': 'table', 'text': 'Status Code | Meaning\n200 | Payment successful\n400 | Invalid request\n401 | Unauthorized\n429 | Rate limit exceeded'}]}
{'tag': 'h1', 'text': 'Payment Service'}
{'tag': 'p', 'text': 'The Payment Service provides APIs for processing customer payments.'}
{'ta

In [44]:
"""
Payment Service
│
├── Authentication
│   └── content
│
├── Error Handling
│   ├── 401 content
│   └── 429 content
│
├── Retry Policy
│   └── content
│
└── Payment Status Codes
    └── table
"""

'\nPayment Service\n│\n├── Authentication\n│   └── content\n│\n├── Error Handling\n│   ├── 401 content\n│   └── 429 content\n│\n├── Retry Policy\n│   └── content\n│\n└── Payment Status Codes\n    └── table\n'

In [77]:
MAX_CHARS = 700

MAX_CHARS = 700


def split_content(content: list[str]) -> list[str]:

    chunks = []
    current = ""

    for text in content:

        if not current:
            current = text
            continue

        candidate = current + "\n" + text

        if len(candidate) <= MAX_CHARS:
            current = candidate

        else:
            chunks.append(current)
            current = text

    if current:
        chunks.append(current)

    return chunks

In [78]:
def create_chunks(parsed_document: dict) -> list[dict]:

    document_title = parsed_document['title']
    elements = parsed_document['elements']

    chunks = []

    heading_stack = []
    current_content = []
    chunk_number = 0

    chunk_number = 0

    def save_chunk():
        nonlocal chunk_number

        if not current_content:
            return

        chunk_number += 1

        section = " > ".join(heading_stack)

        content_chunks = split_content(current_content)
        for content in content_chunks:
            chunks.append({
                "content":"\n".join(current_content),
                "metadata":{
                            "document":document_title,
                            "section": section,
                            "chunk_id":f"chunk_{chunk_number}"
                        }
            })

    for element in elements:

        tag = element["tag"]
        text = element["text"]

        if tag in ["h1", "h2", "h3"]:

                save_chunk()
                current_content = []

                level = int(tag[1])

                heading_stack = heading_stack[:level - 1]
                heading_stack.append(text)

        else:
            current_content.append(text)

    # Save the final chunk
    save_chunk()
    return chunks

In [80]:
chunks = create_chunks(parsed_document=result)

for chunk in chunks:

    print("=" * 70)

    print("CHUNK ID:", chunk["metadata"]["chunk_id"])

    print("SECTION:", chunk["metadata"]["section"])

    print("CONTENT:")
    print(chunk["content"])

    print()

CHUNK ID: chunk_1
SECTION: Payment Service
CONTENT:
The Payment Service provides APIs for processing customer payments.

CHUNK ID: chunk_2
SECTION: Payment Service > Authentication
CONTENT:
The Payment API uses OAuth2 authentication.
Access tokens expire after 60 minutes.

CHUNK ID: chunk_3
SECTION: Payment Service > Error Handling
CONTENT:
HTTP 401 indicates an invalid or expired token.
HTTP 429 indicates that the rate limit has been exceeded.

CHUNK ID: chunk_4
SECTION: Payment Service > Retry Policy
CONTENT:
Requests returning HTTP 429 can be retried using exponential backoff.

CHUNK ID: chunk_5
SECTION: Payment Service > Payment Status Codes
CONTENT:
Status Code | Meaning
200 | Payment successful
400 | Invalid request
401 | Unauthorized
429 | Rate limit exceeded



In [ ]:
"""
Quality for retrival chunks

MMR  = Maximal Marginal Relevance

It is a retrieval strategy used in RAG to select results that are both relevant
to the query and diverse from each other. 

Relevance to query
        +
Diversity from already selected chunks
"""